# Agent Loop Function Calling - Databricks Foundation Model API

Ez a notebook az Ollama-alapú agent loop átirata **Databricks Foundation Model API** és **MLflow tracking** használatával.

## Főbb különbségek az Ollama verziótól:

1. **LLM Provider**: Ollama helyett Databricks Foundation Model API (WorkspaceClient)
2. **Modell**: `databricks-qwen3-next-80b-a3b-instruct` (Qwen3 80B model Databricks-en)
3. **API formátum**: Databricks ChatMessage objektumok és serving_endpoints.query() használata
4. **Tool calling**: Databricks-kompatibilis function calling formátum
5. **Tracking**: LangSmith helyett **MLflow** experiment tracking

## MLflow Tracking tartalma:

* **Paraméterek**: question, model, max_iterations, tool names, tool arguments
* **Metrikák**: iterations, tool_calls, latency, token counts, prices
* **Tagek**: agent_type, status
* **Artifactok**: final_answer.txt, conversation_history.json

## Eredeti oktatási célú kód:
`/Users/kadarferi@gmail.com/langchain-course/2_agent_loop_raw_function_calling.py`

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage
import mlflow
import json
import time

MAX_ITERATIONS = 10
MODEL = "databricks-qwen3-next-80b-a3b-instruct"  # Databricks Foundation Model

# Initialize Databricks Workspace Client
w = WorkspaceClient()

# MLflow experiment setup
mlflow.set_experiment("/Users/kadarferi@gmail.com/agent-loop-experiments")

In [0]:
# --- Tools (ugyanazok mint az eredeti verzióban) ---

def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    result = prices.get(product, 0)
    
    # MLflow logging
    mlflow.log_param(f"tool_call_product", product)
    mlflow.log_metric(f"product_price", result)
    
    return result


def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.
    Available tiers: bronze, silver, gold."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    final_price = round(price * (1 - discount / 100), 2)
    
    # MLflow logging
    mlflow.log_param(f"discount_tier", discount_tier)
    mlflow.log_metric(f"original_price", price)
    mlflow.log_metric(f"discount_percentage", discount)
    mlflow.log_metric(f"final_price", final_price)
    
    return final_price

In [0]:
# Manuálisan definiált JSON schema a function calling számára
# (Databricks Foundation Model API OpenAI-kompatibilis formátumot használ)

tools_for_llm = [
    {
        "type": "function",
        "function": {
            "name": "get_product_price",
            "description": "Look up the price of a product in the catalog.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {
                        "type": "string",
                        "description": "The product name, e.g. 'laptop', 'headphones', 'keyboard'",
                    },
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount",
            "description": "Apply a discount tier to a price and return the final price. Available tiers: bronze, silver, gold.",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {"type": "number", "description": "The original price"},
                    "discount_tier": {
                        "type": "string",
                        "description": "The discount tier: 'bronze', 'silver', or 'gold'",
                    },
                },
                "required": ["price", "discount_tier"],
            },
        },
    },
]

In [0]:
# Databricks Foundation Model API hívás MLflow logging-gal
import requests

def databricks_chat_traced(messages, iteration):
    """Call Databricks Foundation Model API with tool support using REST API."""
    # Get workspace URL and token
    workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
    
    # Endpoint URL
    url = f"{workspace_url}/serving-endpoints/{MODEL}/invocations"
    
    # Headers
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    # Request body
    body = {
        "messages": messages,
        "max_tokens": 10000,
        "tools": tools_for_llm
    }
    
    # Mérés késleltetés
    start_time = time.time()
    
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    
    end_time = time.time()
    latency = end_time - start_time
    
    # Parse response
    response_data = response.json()
    
    # MLflow logging
    mlflow.log_metric(f"iteration_{iteration}_latency_seconds", latency)
    mlflow.log_metric(f"iteration_{iteration}_input_messages", len(messages))
    
    # Usage statistics (ha elérhető)
    if "usage" in response_data:
        usage = response_data["usage"]
        mlflow.log_metric(f"iteration_{iteration}_prompt_tokens", usage.get("prompt_tokens", 0))
        mlflow.log_metric(f"iteration_{iteration}_completion_tokens", usage.get("completion_tokens", 0))
        mlflow.log_metric(f"iteration_{iteration}_total_tokens", usage.get("total_tokens", 0))
    
    return response_data

In [0]:
def run_agent(question: str):
    """Agent loop with MLflow tracking."""
    
    # MLflow run indítása
    with mlflow.start_run(run_name=f"agent_run_{int(time.time())}") as run:
        
        # Paraméterek loggolása
        mlflow.log_param("question", question)
        mlflow.log_param("model", MODEL)
        mlflow.log_param("max_iterations", MAX_ITERATIONS)
        mlflow.set_tag("agent_type", "function_calling")
        
        tools_dict = {
            "get_product_price": get_product_price,
            "apply_discount": apply_discount,
        }

        print(f"Question: {question}")
        print(f"MLflow Run ID: {run.info.run_id}")
        print("=" * 60)

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful shopping assistant. "
                    "You have access to a product catalog tool "
                    "and a discount tool.\n\n"
                    "STRICT RULES — you must follow these exactly:\n"
                    "1. NEVER guess or assume any product price. "
                    "You MUST call get_product_price first to get the real price.\n"
                    "2. Only call apply_discount AFTER you have received "
                    "a price from get_product_price. Pass the exact price "
                    "returned by get_product_price — do NOT pass a made-up number.\n"
                    "3. NEVER calculate discounts yourself using math. "
                    "Always use the apply_discount tool.\n"
                    "4. If the user does not specify a discount tier, "
                    "ask them which tier to use — do NOT assume one."
                ),
            },
            {"role": "user", "content": question},
        ]
        
        tool_calls_count = 0
        start_time = time.time()

        for iteration in range(1, MAX_ITERATIONS + 1):
            print(f"\n--- Iteration {iteration} ---")

            # Databricks Foundation Model API hívás
            response = databricks_chat_traced(messages=messages, iteration=iteration)
            
            # Parse response (REST API JSON formátum)
            choices = response.get("choices", [])
            if not choices:
                print("ERROR: No response choices returned")
                mlflow.set_tag("status", "error")
                mlflow.log_param("error_message", "No response choices returned")
                return None
                
            ai_message = choices[0].get("message", {})
            tool_calls = ai_message.get("tool_calls", [])

            # Ha nincs tool call, ez a végső válasz
            if not tool_calls:
                final_answer = ai_message.get("content", "")
                print(f"\nFinal Answer: {final_answer}")
                
                # Végső metrikák
                total_time = time.time() - start_time
                mlflow.log_metric("total_iterations", iteration)
                mlflow.log_metric("total_tool_calls", tool_calls_count)
                mlflow.log_metric("total_time_seconds", total_time)
                mlflow.set_tag("status", "success")
                
                # Végső válasz mentése
                mlflow.log_text(final_answer, "final_answer.txt")
                
                # Üzenetek történet mentése
                messages_log = json.dumps(messages, indent=2, ensure_ascii=False)
                mlflow.log_text(messages_log, "conversation_history.json")
                
                return final_answer

            # Csak az ELSŐ tool call feldolgozása — egy tool iterációnként
            tool_call = tool_calls[0]
            tool_name = tool_call.get("function", {}).get("name", "")
            tool_args_str = tool_call.get("function", {}).get("arguments", "{}")
            tool_call_id = tool_call.get("id", "")
            
            # Parse arguments (JSON string formátumban jön)
            tool_args = json.loads(tool_args_str) if isinstance(tool_args_str, str) else tool_args_str

            print(f"  [Tool Selected] {tool_name} with args: {tool_args}")
            
            # Tool call logging
            tool_calls_count += 1
            mlflow.log_metric(f"iteration_{iteration}_tool_calls", 1)
            mlflow.log_param(f"iteration_{iteration}_tool_name", tool_name)
            mlflow.log_param(f"iteration_{iteration}_tool_args", json.dumps(tool_args))

            tool_to_use = tools_dict.get(tool_name)
            if tool_to_use is None:
                error_msg = f"Tool '{tool_name}' not found"
                mlflow.set_tag("status", "error")
                mlflow.log_param("error_message", error_msg)
                raise ValueError(error_msg)

            # Tool végrehajtása
            observation = tool_to_use(**tool_args)
            print(f"  [Tool Result] {observation}")
            
            mlflow.log_param(f"iteration_{iteration}_tool_result", str(observation))

            # Üzenetek frissítése
            messages.append({
                "role": "assistant",
                "content": ai_message.get("content", "") or "",
                "tool_calls": [{"id": tool_call_id, "type": "function", "function": {"name": tool_name, "arguments": tool_args_str}}]
            })
            messages.append({
                "role": "tool",
                "content": str(observation),
                "tool_call_id": tool_call_id
            })

        # Max iteráció elérése
        print("ERROR: Max iterations reached without a final answer")
        mlflow.set_tag("status", "max_iterations_reached")
        mlflow.log_metric("total_iterations", MAX_ITERATIONS)
        mlflow.log_metric("total_tool_calls", tool_calls_count)
        
        # Üzenetek történet mentése
        messages_log = json.dumps(messages, indent=2, ensure_ascii=False)
        mlflow.log_text(messages_log, "conversation_history_incomplete.json")
        
        return None

In [0]:
# Teszt futtatás
print("Hello Databricks Agent (Function Calling)!")
print()
result = run_agent("What is the price of a laptop after applying a gold discount?")

In [0]:
# MLflow Experiment megtekintése

# Aktuális experiment adatai
experiment = mlflow.get_experiment_by_name("/Users/kadarferi@gmail.com/agent-loop-experiments")

if experiment:
    print(f"Experiment ID: {experiment.experiment_id}")
    print(f"Experiment Name: {experiment.name}")
    print(f"Artifact Location: {experiment.artifact_location}")
    print(f"\nMLflow UI: https://<workspace-url>/#mlflow/experiments/{experiment.experiment_id}")
    
    # Legutóbbi runok lekérdezése
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=5
    )
    
    print(f"\nLegutóbbi {len(runs)} run:")
    display(runs[['run_id', 'start_time', 'params.question', 'metrics.total_iterations', 'metrics.total_tool_calls', 'tags.status']])
else:
    print("Experiment még nem létezik. Futtasd először az agent-et!")